# <font color="#49699E" size=40>Collecting Data from the Web: Scraping</font>

# LEARNING OBJECTIVES
# LEARNING MATERIALS


# INTRODUCTION


# AN HTML AND CSS PRIMER FOR WEB SCRAPERS


## DEVELOPING YOUR FIRST WEB SCRAPER


### Studying Website Source Code with Developer Tools


### Coding a Web Scraper for a Single Page


In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = 'https://www.theguardian.com/politics/2019/aug/02/europes-view-on-boris-johnson'

r = requests.get(url)
soup = BeautifulSoup(r.content, 'lxml')

In [2]:
article_title = soup.findAll('title')[0].text.replace('\n', '')
print(article_title)

Charming but dishonest and duplicitous: Europe's verdict on Boris Johnson | Boris Johnson | The Guardian


/var/folders/b5/h41fdq7j1ys3hmsdxf5sk9jh0000gp/T/ipykernel_46593/1612112981.py:1: DeprecationWarning: Call to deprecated method findAll. (Replaced by find_all) -- Deprecated since version 4.0.0.
  article_title = soup.findAll('title')[0].text.replace('\n', '')


In [3]:
paragraphs = soup.findAll('p')

/var/folders/b5/h41fdq7j1ys3hmsdxf5sk9jh0000gp/T/ipykernel_46593/68030946.py:1: DeprecationWarning: Call to deprecated method findAll. (Replaced by find_all) -- Deprecated since version 4.0.0.
  paragraphs = soup.findAll('p')


In [4]:
paragraphs[8].text

'But his very presence in No 10 showed Johnson was not the bumbler the continental media like to portray him as, Gattolin said, adding: “He pretends he’s a bull in a china shop – but he knew how to get in, by the front door. He’s playing a game, and thus far you’d have to say he’s playing it pretty well.”'

In [5]:
all_text = " ".join(para.text for para in paragraphs)

In [6]:
def scrape_guardian_stories(url):
    soup = BeautifulSoup(requests.get(url).content, 'lxml')
    article_title = soup.find('title').text.replace('\n', '')
    paras = " ".join(para.text.replace('\n', '') for para in soup.findAll('p'))
    return [article_title, paras]

In [7]:
with open('../data/scraping/guardian_story_links.txt') as f:
    stories = [line.rstrip() for line in f]

scraped = [scrape_guardian_stories(s) for s in stories]
df_scraped = pd.DataFrame(scraped, columns=['Title', 'Article Text'])
print(df_scraped.info())

/var/folders/b5/h41fdq7j1ys3hmsdxf5sk9jh0000gp/T/ipykernel_46593/722351634.py:4: DeprecationWarning: Call to deprecated method findAll. (Replaced by find_all) -- Deprecated since version 4.0.0.
  paras = " ".join(para.text.replace('\n', '') for para in soup.findAll('p'))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Title         5 non-null      object
 1   Article Text  5 non-null      object
dtypes: object(2)
memory usage: 212.0+ bytes
None


### Working with Many Webpages


**2026 update:** The original example scraped the United Nations Partnerships for SDGs platform at `sustainabledevelopment.un.org`. That platform has since been retired and the host now returns HTTP 403 to automated requests, so the `id='headline'` and `id='intro'` elements the scraper looked for no longer exist. The scraping pattern below is unchanged so you can still study it, but the loop is now bounded by `max_attempts` and will collect no records against the live site. Point `scrape_UNSD_project` at a page you are permitted to scrape to see it return data.

In [8]:
def scrape_UNSD_project(url):
    result = requests.get(url, timeout=15)
    if result.ok:
        soup = BeautifulSoup(result.content, 'lxml')
        headline = soup.find(id='headline').getText()
        intro = " ".join(
            [segment for segment in soup.find(id='intro').stripped_strings])
        return [headline, intro]
    else:
        return None

In [9]:
base_url = "https://sustainabledevelopment.un.org/partnership/?p={}"
starting_number = 30000
target_records = 30

In [10]:
scraped = []

current_number = starting_number
attempts = 0
max_attempts = 15  # 2026 update: bound the loop so it terminates if the site returns no records

while len(scraped) < target_records and attempts < max_attempts:
    url = base_url.format(current_number)
    try:
        output = scrape_UNSD_project(url)
        if output is not None:
            print(f"scraping {current_number}")
            scraped.append(output)
    except AttributeError:
        pass
    current_number += 1
    attempts += 1

df_scraped = pd.DataFrame(scraped, columns=['Headline', 'Introduction'])

print(df_scraped.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Headline      0 non-null      object
 1   Introduction  0 non-null      object
dtypes: object(2)
memory usage: 132.0+ bytes
None


## ETHICAL AND LEGAL ISSUES IN WEB SCRAPING


# CONCLUSION
## Key Points 
